# 03 — A2A Protocol + Hybrid Architecture
**Study Notebook 3 of 3 · Estimated time: 45 minutes**

By the end of this notebook you will:
- Understand the A2A (Agent-to-Agent) protocol: Agent Cards, Tasks, Artifacts
- Build 3 specialist A2A agents (one wraps a smolagents `CodeAgent` inside)
- Wire all three into LangGraph to build the **FOSS Contribution Matchmaker** pipeline

```
What we're building:

GitHub profile URL + interests
    │
    ▼  LangGraph: parse input, analyse profile
    │
    ▼  A2A: RepoFinderAgent    ← searches GitHub API
    │
    ▼  A2A: ComplexityRaterAgent  ← rates repos for your level
    │
    ▼  A2A: OnboardingGuideAgent  ← smolagents CodeAgent inside!
    │        fetches CONTRIBUTING.md + open issues
    │
    ▼  LangGraph: format plan
    │
    ▼  "Your First PR" personalised plan
```

---
## Step 0 — Install & API Keys

In [ ]:
!pip install -q "smolagents[toolkit]" litellm langgraph requests

In [ ]:
import os, getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")

# Optional but recommended: avoids GitHub rate limits during live demos
# Get one free at github.com -> Settings -> Developer settings -> Personal access tokens (no scopes needed)
github_token = getpass.getpass("GitHub token (press Enter to skip): ")
if github_token:
    os.environ["GITHUB_TOKEN"] = github_token

print("Setup complete.")

In [ ]:
import requests, json, uuid, base64, re
from dataclasses import dataclass, field
from typing import Any, Optional, List
from enum import Enum
from smolagents import CodeAgent, LiteLLMModel, tool

model = LiteLLMModel(model_id="groq/llama-3.3-70b-versatile")

def gh_get(url, params=None):
    """GitHub API request with optional auth header."""
    headers = {}
    if os.environ.get("GITHUB_TOKEN"):
        headers["Authorization"] = f"token {os.environ['GITHUB_TOKEN']}"
    return requests.get(url, headers=headers, params=params, timeout=10)

print("Imports ready.")

---
## Part 1 — The A2A Protocol

**A2A (Agent-to-Agent)** is Google's open protocol (April 2025) for agent interoperability.

| Concept | What it is | Real-world analogy |
|---------|-----------|--------------------|
| **Agent Card** | JSON manifest at `/.well-known/agent.json` | A business card for your agent |
| **Task** | Structured message: message + context → artifacts | A work ticket |
| **Artifact** | Structured output from the agent | The completed deliverable |

In production: each agent runs as a separate HTTP service. Here: same interface, in-process, for Colab portability.

In [ ]:
class TaskState(Enum):
    SUBMITTED = "submitted"
    WORKING   = "working"
    COMPLETED = "completed"
    FAILED    = "failed"

@dataclass
class A2ATask:
    """Mirrors the A2A Task object sent between agents."""
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    message: str = ""
    context: dict = field(default_factory=dict)
    state: TaskState = TaskState.SUBMITTED
    artifacts: List[dict] = field(default_factory=list)
    error: Optional[str] = None

class A2AAgent:
    """
    Base class implementing the A2A protocol pattern.
    In production: subclasses run as independent HTTP services.
    The orchestrator calls POST /tasks with an A2ATask payload.
    """

    @property
    def agent_card(self) -> dict:
        """The A2A Agent Card — returned at /.well-known/agent.json in production."""
        raise NotImplementedError

    def send_task(self, message: str, context: dict = None) -> A2ATask:
        """Send a task to this agent. Mirrors HTTP POST /tasks."""
        task = A2ATask(message=message, context=context or {})
        print(f"  [A2A ->] {self.agent_card['name']} | task:{task.id}")
        task.state = TaskState.WORKING
        try:
            result = self._handle(task)
            task.artifacts = [{"type": "data", "data": result}]
            task.state = TaskState.COMPLETED
            print(f"  [A2A <-] {self.agent_card['name']} | task:{task.id} done")
        except Exception as e:
            task.error = str(e)
            task.state = TaskState.FAILED
            print(f"  [A2A x]  {self.agent_card['name']} | task:{task.id} failed: {e}")
        return task

    def _handle(self, task: A2ATask) -> Any:
        raise NotImplementedError

print("A2A base classes defined.")

---
## Part 2 — Build the 3 Specialist Agents

### Agent 1: RepoFinderAgent
Searches GitHub for repos matching a developer's skill profile.

In [ ]:
class RepoFinderAgent(A2AAgent):

    @property
    def agent_card(self):
        return {
            "name": "RepoFinderAgent",
            "description": "Discovers GitHub repositories matching a developer skill profile with open contribution opportunities",
            "version": "1.0.0",
            "capabilities": {"streaming": False, "push_notifications": False},
            "skills": [{
                "id": "find_matching_repos",
                "name": "Find Matching Repositories",
                "description": "Search GitHub for repos with good-first-issues matching given languages and level",
                "input_modes": ["text"], "output_modes": ["data"]
            }]
        }

    def _handle(self, task: A2ATask) -> list:
        languages  = task.context.get("languages", ["Python"])
        experience = task.context.get("experience", "intermediate")
        star_range = "100..2000" if experience == "beginner" else "500..50000"

        results = []
        for lang in languages[:2]:
            query = f"language:{lang} good-first-issues:>3 stars:{star_range}"
            resp  = gh_get(
                "https://api.github.com/search/repositories",
                params={"q": query, "sort": "updated", "per_page": 5}
            )
            if resp.status_code == 200:
                for repo in resp.json().get("items", []):
                    results.append({
                        "full_name":   repo["full_name"],
                        "description": (repo.get("description") or "")[:120],
                        "stars":       repo["stargazers_count"],
                        "language":    repo.get("language", ""),
                        "open_issues": repo["open_issues_count"],
                        "topics":      repo.get("topics", [])[:5]
                    })
        return results[:6]

# Show the Agent Card
print(json.dumps(RepoFinderAgent().agent_card, indent=2))

In [ ]:
task = RepoFinderAgent().send_task(
    message="Find repos for a Python developer",
    context={"languages": ["Python"], "experience": "beginner"}
)
print(f"State: {task.state.value}")
for repo in task.artifacts[0]["data"]:
    print(f"  {repo['full_name']} ({repo['stars']} stars) — {repo['description'][:60]}")

### Agent 2: ComplexityRaterAgent
Rates a repository's complexity using deterministic heuristics — no LLM call needed.

In [ ]:
class ComplexityRaterAgent(A2AAgent):

    @property
    def agent_card(self):
        return {
            "name": "ComplexityRaterAgent",
            "description": "Rates the contributor complexity of a GitHub repository",
            "version": "1.0.0",
            "capabilities": {"streaming": False, "push_notifications": False},
            "skills": [{
                "id": "rate_complexity",
                "name": "Rate Repository Complexity",
                "description": "Analyses repo structure and rates as beginner-friendly / intermediate / advanced",
                "input_modes": ["text"], "output_modes": ["data"]
            }]
        }

    def _handle(self, task: A2ATask) -> dict:
        owner = task.context["owner"]
        repo  = task.context["repo"]

        repo_data  = gh_get(f"https://api.github.com/repos/{owner}/{repo}").json()
        langs_data = gh_get(f"https://api.github.com/repos/{owner}/{repo}/languages").json()

        stars     = repo_data.get("stargazers_count", 0)
        forks     = repo_data.get("forks_count", 0)
        num_langs = len(langs_data) if isinstance(langs_data, dict) else 1

        score = 0
        if stars > 10000: score += 2
        elif stars > 2000: score += 1
        if num_langs > 4: score += 2
        elif num_langs > 2: score += 1
        if forks > 1000: score += 1

        if score <= 1:
            complexity, tip = "beginner-friendly", "Small codebase. Great starting point."
        elif score <= 3:
            complexity, tip = "intermediate", "Moderate complexity. Some experience helpful."
        else:
            complexity, tip = "advanced", "Large project. Better after a few contributions elsewhere."

        return {
            "repo":       f"{owner}/{repo}",
            "complexity": complexity,
            "tip":        tip,
            "stars":      stars,
            "languages":  list(langs_data.keys())[:5] if isinstance(langs_data, dict) else []
        }

task = ComplexityRaterAgent().send_task(
    message="Rate smolagents",
    context={"owner": "huggingface", "repo": "smolagents"}
)
print(json.dumps(task.artifacts[0]["data"], indent=2))

### Agent 3: OnboardingGuideAgent

This is where it gets interesting. This A2A agent **internally wraps a smolagents `CodeAgent`**.

```
LangGraph node
    └── calls OnboardingGuideAgent.send_task()  [A2A]
            └── runs CodeAgent.run()            [smolagents]
                    └── writes Python to call:
                            fetch_contributing_guide()  [tool]
                            fetch_starter_issues()      [tool]
                                └── GitHub API
```

In [ ]:
@tool
def fetch_contributing_guide(owner: str, repo: str) -> str:
    """Fetches the CONTRIBUTING.md from a GitHub repository.

    Args:
        owner: GitHub username or organisation.
        repo: Repository name.
    """
    for path in ["CONTRIBUTING.md", ".github/CONTRIBUTING.md", "docs/contributing.md"]:
        resp = gh_get(f"https://api.github.com/repos/{owner}/{repo}/contents/{path}")
        if resp.status_code == 200:
            raw = base64.b64decode(resp.json()["content"]).decode("utf-8", errors="ignore")
            clean = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', raw)
            clean = re.sub(r'[`#*<>!]', '', clean)
            return clean.strip()[:1200]
    return "No CONTRIBUTING.md found. Check the README for how to contribute."

@tool
def fetch_starter_issues(owner: str, repo: str) -> str:
    """Fetches open 'good first issue' and 'help wanted' issues from a GitHub repository.

    Args:
        owner: GitHub username or organisation.
        repo: Repository name.
    """
    found = []
    for label in ["good first issue", "help wanted"]:
        resp = gh_get(
            f"https://api.github.com/repos/{owner}/{repo}/issues",
            params={"labels": label, "state": "open", "per_page": 3}
        )
        if resp.status_code == 200:
            for issue in resp.json():
                if isinstance(issue, dict):
                    found.append(f"[{label}] #{issue['number']}: {issue['title']}")
    return "\n".join(found) if found else "No labelled starter issues found — browse open issues on GitHub."

print("Inner tools ready.")

In [ ]:
class OnboardingGuideAgent(A2AAgent):
    """An A2A agent that runs a smolagents CodeAgent internally."""

    def __init__(self, llm_model):
        self._inner_agent = CodeAgent(
            tools=[fetch_contributing_guide, fetch_starter_issues],
            model=llm_model
        )

    @property
    def agent_card(self):
        return {
            "name": "OnboardingGuideAgent",
            "description": "Generates a personalised first-contribution guide using an inner smolagents CodeAgent",
            "version": "1.0.0",
            "capabilities": {"streaming": False, "push_notifications": False},
            "skills": [{
                "id": "generate_onboarding",
                "name": "Generate Onboarding Guide",
                "description": "Fetches CONTRIBUTING.md and open issues, returns a step-by-step first-PR plan",
                "input_modes": ["text"], "output_modes": ["text"]
            }]
        }

    def _handle(self, task: A2ATask) -> str:
        owner  = task.context["owner"]
        repo   = task.context["repo"]
        skills = task.context.get("skills", ["Python"])

        # The CodeAgent writes Python to call the tools — watch it happen live
        return self._inner_agent.run(
            f"Generate a beginner-friendly first-contribution guide for {owner}/{repo}. "
            f"The developer knows: {', '.join(skills)}. "
            "Use the tools to fetch the contributing guide and starter issues. "
            "Return: 1) Setup steps (3 bullets), 2) Best first issue to tackle and why, "
            "3) Which files are likely to need editing."
        )

onboarding = OnboardingGuideAgent(model)

# Run it — watch the LLM write Python live
task = onboarding.send_task(
    message="Onboard me into smolagents",
    context={"owner": "huggingface", "repo": "smolagents", "skills": ["Python"]}
)
print("\n" + task.artifacts[0]["data"])

---
## Part 3 — LangGraph Orchestration

Now wire all three A2A agents into a state graph. Notice: **LangGraph never calls an LLM** — it only routes Python functions. The LLM is isolated inside `OnboardingGuideAgent`.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class MatchmakerState(TypedDict):
    github_url:        str
    interests:         str
    username:          str
    skill_profile:     dict
    candidate_repos:   list
    rated_repos:       list
    onboarding_guides: list
    contribution_plan: str

def parse_input_node(state: MatchmakerState) -> dict:
    url      = state["github_url"].rstrip("/")
    username = url.replace("https://github.com/", "").split("/")[0]
    print(f"[LangGraph] username = {username}")
    return {"username": username}

def analyze_profile_node(state: MatchmakerState) -> dict:
    resp  = gh_get(
        f"https://api.github.com/users/{state['username']}/repos",
        params={"sort": "updated", "per_page": 15}
    )
    repos = resp.json() if resp.status_code == 200 else []

    lang_count = {}
    for r in repos:
        lang = r.get("language")
        if lang:
            lang_count[lang] = lang_count.get(lang, 0) + 1

    top_langs  = sorted(lang_count, key=lang_count.get, reverse=True)[:3] or ["Python"]
    total      = len(repos)
    experience = "beginner" if total < 6 else "intermediate" if total < 20 else "experienced"

    profile = {
        "username":      state["username"],
        "top_languages": top_langs,
        "experience":    experience,
        "interests":     [i.strip() for i in state.get("interests", "").split(",") if i.strip()]
    }
    print(f"[LangGraph] Profile: {top_langs} | {experience}")
    return {"skill_profile": profile}

def repo_discovery_node(state: MatchmakerState) -> dict:
    print("[LangGraph] -> RepoFinderAgent (A2A)")
    task = RepoFinderAgent().send_task(
        message="Find repos matching this developer",
        context={
            "languages":  state["skill_profile"]["top_languages"],
            "experience": state["skill_profile"]["experience"]
        }
    )
    candidates = task.artifacts[0]["data"] if task.state == TaskState.COMPLETED else []
    return {"candidate_repos": candidates}

def complexity_rating_node(state: MatchmakerState) -> dict:
    print("[LangGraph] -> ComplexityRaterAgent (A2A)")
    rater = ComplexityRaterAgent()
    rated = []
    for repo in state["candidate_repos"][:4]:
        owner, repo_name = repo["full_name"].split("/")
        task = rater.send_task(
            message=f"Rate {repo['full_name']}",
            context={"owner": owner, "repo": repo_name}
        )
        if task.state == TaskState.COMPLETED:
            r = task.artifacts[0]["data"]
            r["description"] = repo.get("description", "")
            rated.append(r)

    experience = state["skill_profile"]["experience"]
    target     = "beginner-friendly" if experience == "beginner" else "intermediate"
    matched    = [r for r in rated if r["complexity"] == target] or rated
    return {"rated_repos": matched}

def onboarding_node(state: MatchmakerState) -> dict:
    print("[LangGraph] -> OnboardingGuideAgent (A2A -> smolagents inside)")
    guide_agent = OnboardingGuideAgent(model)
    skills      = state["skill_profile"]["top_languages"]
    guides      = []
    for repo in state["rated_repos"][:2]:
        owner, repo_name = repo["repo"].split("/")
        task = guide_agent.send_task(
            message=f"Onboard into {repo['repo']}",
            context={"owner": owner, "repo": repo_name, "skills": skills}
        )
        if task.state == TaskState.COMPLETED:
            guides.append({
                "repo":       repo["repo"],
                "complexity": repo["complexity"],
                "tip":        repo["tip"],
                "guide":      task.artifacts[0]["data"]
            })
    return {"onboarding_guides": guides}

def format_plan_node(state: MatchmakerState) -> dict:
    p      = state["skill_profile"]
    guides = state["onboarding_guides"]
    lines  = [
        "=" * 54,
        f"  Your First PR Plan  --  @{p['username']}",
        "=" * 54,
        f"Skills: {', '.join(p['top_languages'])}  |  Level: {p['experience']}",
        ""
    ]
    for i, g in enumerate(guides, 1):
        lines += [
            f"--- Opportunity {i}: {g['repo']} [{g['complexity']}] ---",
            g["tip"], "",
            g["guide"], ""
        ]
    lines.append("Powered by: LangGraph + A2A + smolagents + Llama 3.3 on Groq")
    return {"contribution_plan": "\n".join(lines)}

print("All nodes defined.")

In [ ]:
builder = StateGraph(MatchmakerState)
for name, fn in [
    ("parse_input",       parse_input_node),
    ("analyze_profile",   analyze_profile_node),
    ("repo_discovery",    repo_discovery_node),
    ("complexity_rating", complexity_rating_node),
    ("onboarding",        onboarding_node),
    ("format_plan",       format_plan_node),
]:
    builder.add_node(name, fn)

builder.set_entry_point("parse_input")
builder.add_edge("parse_input",       "analyze_profile")
builder.add_edge("analyze_profile",   "repo_discovery")
builder.add_edge("repo_discovery",    "complexity_rating")
builder.add_edge("complexity_rating", "onboarding")
builder.add_edge("onboarding",        "format_plan")
builder.add_edge("format_plan",       END)

matchmaker = builder.compile()
print(matchmaker.get_graph().draw_mermaid())

---
## Part 4 — Run the Full Pipeline

In [ ]:
result = matchmaker.invoke({
    "github_url":        "https://github.com/torvalds",  # change to any public profile
    "interests":         "web, APIs",
    "username":          "",
    "skill_profile":     {},
    "candidate_repos":   [],
    "rated_repos":       [],
    "onboarding_guides": [],
    "contribution_plan": ""
})
print(result["contribution_plan"])

---
## Summary

| Step | Framework | What happened | LLM call? |
|------|-----------|---------------|-----------|
| parse_input | LangGraph | Extracted username from URL | No |
| analyze_profile | LangGraph | Fetched repos, built skill profile | No |
| repo_discovery | A2A RepoFinderAgent | GitHub API search | No |
| complexity_rating | A2A ComplexityRaterAgent | Python heuristic on repo metadata | No |
| onboarding | A2A OnboardingGuideAgent + smolagents CodeAgent | LLM wrote Python, called 2 tools | **Yes — once per repo** |
| format_plan | LangGraph | String formatting | No |

**Total LLM calls: 2** (one per onboarding guide). All routing and filtering: pure Python.

**Next:** `foss_demo.ipynb` — the polished live demo version with the audience challenge.